In [20]:
import os
import ast
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import joblib

In [21]:
model = joblib.load(r"C:\Users\oozgen\Desktop\Code_Dataset\5. work model\RF_acc_dpc_bestModel.joblib")

In [22]:
# Veri setini yükle
df = pd.read_csv(r"C:\Users\oozgen\OneDrive - Nobel\Belgeler\Tez\v1\3. Gereç ve Yöntem\Code_Dataset\2. feature selection\prediction-set-features.csv")

# Gruplama işlemi yap ve ilk örneği koru
groups = df.groupby("sequence")
df = groups.first().reset_index()

# Rastgele 10.000 satır seç
df_sampled = df.sample(n=10000, random_state=42)

# 'sequence' sütununu kaydet
sequence_column = df_sampled["sequence"]

# 'sequence' ve 'numeric_sequence' kolonlarını çıkar
df_sampled = df_sampled.drop(["sequence", "numeric_sequence"], axis=1)

# AAC ve DPC sütunlarını işle ve kaydet
aac_dpc_columns = pd.DataFrame()
for col in ["AAC", "DPC"]:
    col_df = df_sampled[col].apply(lambda x: pd.Series(ast.literal_eval(x)))
    col_df.columns = [f"{col}_{str(c)}" for c in col_df.columns]
    aac_dpc_columns = pd.concat([aac_dpc_columns, col_df], axis=1)
    df_sampled = pd.concat([df_sampled.drop(col, axis=1), col_df], axis=1)

# Eksik değerleri doldur
df_sampled.fillna(0, inplace=True)

In [23]:
# Özellik ölçeklendirme
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_sampled)

# Tahmin yap
predictions = model.predict(X_scaled)

# Tahminleri ve sequence kolonunu birleştir
predictions_df = pd.DataFrame(predictions, columns=["anticancer_predicted"])
df_combined = pd.concat([sequence_column.reset_index(drop=True), predictions_df, aac_dpc_columns.reset_index(drop=True)], axis=1)

# Sonuçları kaydet
df_combined.to_csv("predictions.csv", index=False)